# Setup

In [1]:
from huggingface_hub import login, logout
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from datasets import load_dataset
import numpy as np

/home/peter/Desktop/Studies/Thesis/ThesisProject/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv() 
HF_login = os.getenv("HF_KEY")

login(HF_login)

# fetch data and inspect

In [3]:
import torch
import soundfile as sf
import torchaudio

In [4]:
from datasets import load_dataset
train = load_dataset("google/fleurs", "ga_ie", split="train")
val = load_dataset("google/fleurs", "ga_ie", split="validation")
test = load_dataset("google/fleurs", "ga_ie", split="test")

Using the latest cached version of the dataset since google/fleurs couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ga_ie' at /home/peter/.cache/huggingface/datasets/google___fleurs/ga_ie/2.0.0/80cb68d1b4d319aefbd8ea302274d3950d95f6242f0742c1452d1545c80a2d5f (last modified on Tue Mar  4 15:00:34 2025).
Using the latest cached version of the dataset since google/fleurs couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ga_ie' at /home/peter/.cache/huggingface/datasets/google___fleurs/ga_ie/2.0.0/80cb68d1b4d319aefbd8ea302274d3950d95f6242f0742c1452d1545c80a2d5f (last modified on Tue Mar  4 15:00:34 2025).
Using the latest cached version of the dataset since google/fleurs couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ga_ie' at /home/peter/.cache/huggingface/datasets/google___fleurs/ga_ie/2.0.0/80cb68d1b4d319aefbd8ea302274d3950d95f6242f0742c1452d1545c80a2d5f (last modifie

In [ ]:
train[0]

{'id': 571,
 'num_samples': 172800,
 'path': '/home/peter/.cache/huggingface/datasets/downloads/extracted/a870d8d6658a3c263aa78e1d5ab46b448872cfe59db75af5d50a98907d873444/10009174761044778838.wav',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7297782072c0>,
 'transcription': 'nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe tugtar daonra monómorfach orthu',
 'raw_transcription': 'Nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe, tugtar daonra monómorfach orthu.',
 'gender': 0,
 'lang_id': 27,
 'language': 'Irish',
 'lang_group_id': 0}

In [ ]:
audio_input = train[0]["audio"]

In [ ]:
train[0]

{'id': 571,
 'num_samples': 172800,
 'path': '/home/peter/.cache/huggingface/datasets/downloads/extracted/a870d8d6658a3c263aa78e1d5ab46b448872cfe59db75af5d50a98907d873444/10009174761044778838.wav',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x72948b8e3470>,
 'transcription': 'nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe tugtar daonra monómorfach orthu',
 'raw_transcription': 'Nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe, tugtar daonra monómorfach orthu.',
 'gender': 0,
 'lang_id': 27,
 'language': 'Irish',
 'lang_group_id': 0}

In [ ]:
from IPython.display import Audio

# Play some sample. (audio doesn't work for me period in vscode)
Audio(filename='/home/peter/Desktop/Studies/Thesis/ThesisProject/data/fleurs/audio/train/normalized/7141226710506360.wav', rate=16000, autoplay=True)

# load g2p dict

In [5]:
import pandas as pd

In [6]:
g2p_path="../../../data/abair_g2p/ulster.tsv"
g2p_file = pd.read_csv(g2p_path,sep="\t", names=["word","phonemes"])
# turn df into dict for simple lookup
g2p_dict = g2p_file.set_index("word")["phonemes"].to_dict()

# Add phoneme transcriptions

In [7]:
import pandas as pd

## mapping function


In [23]:
!mfa g2p {input_file} {model_path} {output_file}   

/bin/bash: line 1: mfa: command not found


In [ ]:
# helper for mapping function
import tempfile
import os

def mfa_fallback(words, model_path="../../../data/abair_g2p/g2p-ulster.zip"):
    """
    Make MFA g2p to generate phonemes for OOV words
    Args:
        words (_type_): _description_
        model_path (str, optional): _description_. Defaults to "irish_g2p.zip".

    Returns:
        _type_: _description_
    """
    # create temporary files for input/output
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
        input_file = f.name
        for word in words:
            f.write(word + "\n")
    
    output_file = tempfile.mkstemp(suffix='.txt')
    
    try:
        # Run MFA to get phonemes. mfa g2p [OPTIONS] INPUT_PATH G2P_MODEL_PATH OUTPUT_PATH
        !mfa g2p {input_file} {model_path} {output_file}   
    
        # Load results into fallback dict
        ipa_fallbacks = {}
        with open(output_file, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) == 2:
                    word, ipa = parts
                    ipa_fallbacks[word] = ipa
        
        return ipa_fallbacks
    
    finally:
        # Clean up temp files
        if os.path.exists(input_file):
            os.remove(input_file)
        if os.path.exists(output_file[1]):
            os.remove(output_file[1])

In [17]:
# The function needs to accept and output a dict
def mfa_g2p(example):
    word_list = [x.strip(" .,!?:;") for x in example["transcription"].split()]

    # fix oovs all at once
    oov_words = [word for word in word_list if word not in g2p_dict]
    ipa_fallbacks = {}
    if oov_words:
        ipa_fallbacks = mfa_fallback(oov_words)

    # lookup and fallback logic
    phoneme_seq = []
    for word in word_list:
        # if in dict...
        if word in g2p_dict: 
            phoneme_seq.append(g2p_dict[word])
        elif word.lower() in g2p_dict:
            phoneme_seq.append(g2p_dict[word.lower()])
        elif word in ipa_fallbacks:
            phoneme_seq.append(ipa_fallbacks[word])
        else:
            phoneme_seq.append("[UNK]")            
            
    example["phonetic"] = " ".join(phoneme_seq)              
    return example

In [ ]:
# old mapping function
#def sent2phones(row):
#    sentence = row["transcription"]
#    words = [x.strip(" .,!?:;") for x in sentence.split()]
#    
##    phoneme_seq = []
#    for word in words:
##        if word in g2p_dict:
#            phoneme_seq.append(g2p_dict[word].replace(" ",""))
#        elif word.lower() in g2p_dict:
#            phoneme_seq.append(g2p_dict[word.lower()].replace(" ",""))
#        else:
#            phoneme_seq.append("[UNK]")
#    
#    return {"phoneme_sentence": "|".join(phoneme_seq)}

In [14]:
mfa_g2p({"transcription":"dia duit"})

{'transcription': 'dia duit', 'phonetic': 'ˈ dʲ ia ˈ d̪ˠ i tʲ'}

In [15]:
train

Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 2845
})

In [22]:
train = train.map(mfa_g2p)

Map:   0%|          | 0/2845 [00:00<?, ? examples/s]

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `mfa g2p /tmp/tmpqc02yei9.txt ../../../data/abair_g2p/g2p-ulster.zip (93, '/tmp/tmpd4u11wts.txt')'


Map:   0%|          | 0/2845 [00:00<?, ? examples/s]


TypeError: expected str, bytes or os.PathLike object, not tuple

In [ ]:
val = val.map(mfa_g2p)

Map: 100%|██████████| 369/369 [00:00<00:00, 578.52 examples/s]


In [ ]:
test = test.map(mfa_g2p)

Map: 100%|██████████| 842/842 [00:01<00:00, 598.16 examples/s]


In [24]:
train[0]

{'id': 571,
 'num_samples': 172800,
 'path': '/home/peter/.cache/huggingface/datasets/downloads/extracted/a870d8d6658a3c263aa78e1d5ab46b448872cfe59db75af5d50a98907d873444/10009174761044778838.wav',
 'audio': {'path': 'train/10009174761044778838.wav',
  'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00030464,
         -0.00026166, -0.00036532]),
  'sampling_rate': 16000},
 'transcription': 'nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe tugtar daonra monómorfach orthu',
 'raw_transcription': 'Nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe, tugtar daonra monómorfach orthu.',
 'gender': 0,
 'lang_id': 27,
 'language': 'Irish',
 'lang_group_id': 0,
 'phoneme_sentence': 'ˈn̻ˠuːɾʲ|ə|ˈvʲiːn̻ˠ|ˈtʲɾʲeː|[UNK]|ˈeɾʲ|ˈl̻ʲehʲ|ˈi|bˠaːɾˠtʲ|ˈeɟ|ˈɡah|ˈd̪ˠinʲə|ˈi|n̻ˠiːn̻ˠɾˠə|ˈaːɾʲihʲə|t̪ˠuɡt̪ˠəɾˠ|ˈd̪ˠiːn̻ˠɾˠə|[UNK]|ˈoɾˠhu'}

## strip suprasegmental features

In [11]:
primary_stress = "ˈ"
secondary_stress = "ˌ"
suprasegmentals = {primary_stress, secondary_stress}
condition = lambda x: x in suprasegmentals  
def strip_suprasegmentals(example):
    phon_string = example['phonetic']
    phone_list = phon_string.split()
    stripped_list = list(filter(lambda x: not condition(x), phone_list))
    stripped_string = " ".join(stripped_list)
    example['phonetic'] = stripped_string
    return example

In [15]:
phonetics = mfa_g2p({"transcription":"dia duit"})
print(phonetics)
print(strip_suprasegmentals(phonetics))

{'transcription': 'dia duit', 'phonetic': 'ˈ dʲ ia ˈ d̪ˠ i tʲ'}
{'transcription': 'dia duit', 'phonetic': 'dʲ ia d̪ˠ i tʲ'}


# Save to disk

In [57]:
fleurs_path = "/home/peter/Desktop/Studies/Thesis/ThesisProject/data/fleurs"
train.save_to_disk(fleurs_path+"/train")
val.save_to_disk(fleurs_path+"/val")
test.save_to_disk(fleurs_path+"/test")


Saving the dataset (2/2 shards): 100%|██████████| 842/842 [00:06<00:00, 123.53 examples/s]


To reload, use...

from datasets import load_from_disk

reloaded_encoded_dataset = load_from_disk("path/of/my/dataset/directory")